### Structured Output

Models can be requested to provide their response in format matching a given schema. This is useful for ensuring the output cen be easily parsed and used in subsrquent processing.Lngachain supports multiple schema types and methods for enforcing structued output

#### Pydantic

Pydantic models provide the richest feature set with field validation,description and nested structures

In [1]:

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing. Check your .env file.")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    reasoning_format="hidden",
)



In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.13', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002212BC29970>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002212BC4EF00>, model_name='openai/gpt-oss-120b', reasoning_format='hidden', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'tit

In [5]:
response = model_with_structure.invoke("Provide me the details of the movie inception")
response

Movie(title='Inception', director='Christopher Nolan', year=2010, genre='Science Fiction')

### Message output alongside parsed Structure

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    year: int = Field(..., description="The release year of the movie")
    genre: str = Field(..., description="The genre of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide me the details of the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fc_62142627-cf32-42c7-9a35-380f355ef55e', 'function': {'arguments': '{"director":"Christopher Nolan","genre":"Science Fiction, Thriller","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 157, 'total_tokens': 272, 'completion_time': 0.244150519, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.046808143, 'prompt_tokens_details': None, 'queue_time': 0.184790958, 'total_time': 0.290958662}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a167e2-4d22-77bf-8ee2-00b3077b7f45-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'genre': 'Science Fiction, Thriller', 'title': 'Inception', 'year': 2010}, 'id': 'fc_62142627-cf32-42c7-

### Nested Structure

In [10]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    director: str
    year: int
    genre: list[str]
    actors: list[Actor]
    budget: float | None = Field(None, description="The budget of the movie")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide me the details of the movie inception")
response

MovieDetails(title='Inception', director='Christopher Nolan', year=2010, genre=['Action', 'Sci-Fi', 'Thriller'], actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Cillian Murphy', role='Robert Fischer')], budget=160000000.0)

### TypeDict
TypeDict provides a simpler alternative using python built-in typing, ideal when you don't need runtime validation

In [13]:
from typing_extensions import TypedDict,Annotated

class Movie(TypedDict):
    title: Annotated[str, "The title of the movie"]
    director: Annotated[str, "The director of the movie"]
    year: Annotated[int, "The release year of the movie"]
    genre: Annotated[str, "The genre of the movie"]

model_withtypedict = model.with_structured_output(Movie)
response = model_withtypedict.invoke("Provide me the details of the movie Avengers Endgame")
response

{'director': 'Anthony Russo, Joe Russo',
 'genre': 'Action, Adventure, Sci-Fi',
 'title': 'Avengers: Endgame',
 'year': 2019}

### DataClasses
A data class is s class typically containing mainly data, Although there aren't really any restriction. You create it using @dataclass decorator

In [15]:

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")




In [ ]:
## Base Model Example
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model = "gpt-4o-mini",
    response_format=ContactInfo
)

result = agent.invoke({"messages":[{"role": "user", "content": "Provide me the contact information for John Doe, email:john.doe@example.com, phone:123-456-7890"}]})

print(result["structured_response"])


name='John Doe' email='john.doe@example.com' phone='123-456-7890'


In [20]:
## TypeDict Example
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: Annotated[str, "The name of the contact"]
    email: Annotated[str, "The email of the contact"]
    phone: Annotated[str, "The phone number of the contact"]

agent = create_agent(
    model = "gpt-4o-mini",
    response_format=ContactInfo
)

result = agent.invoke({"messages":[{"role": "user", "content": "Provide me the contact information for John Doe, email:john.doe@example.com, phone:123-456-7890"}]})

print(result["structured_response"])


{'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}


In [21]:
## DATACLASS 
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str
agent = create_agent(
    model = "gpt-4o-mini",
    response_format=ContactInfo
)   
result = agent.invoke({"messages":[{"role": "user", "content": "Provide me the contact information for John Doe, email:john.doe@example.com, phone:123-456-7890"}]})
print(result["structured_response"])

ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')
